# 5단계: 평가
평가 코드는 `src/evaluate.py` 에 있다. 이 노트북은 GitHub 에서 코드를 받아 **실행만** 한다.
(코드를 고칠 땐 랩탑에서 고치고 push → 여기서 `git pull`)

- 런타임: **A100 GPU** (생성만 하므로 40GB 도 충분). 🔑 Secrets에 `HF_TOKEN` 등록
- 생성 결과(`preds.jsonl`)와 채점 결과는 Drive 의 `MyDrive/octopus_v2/runs/eval/` 에 저장 → 랩탑에서도 다시 채점 가능
- 생성이 중간에 끊기면 같은 칸을 다시 실행하면 남은 것만 이어서 한다
- 비교 기준: 논문 표 1 Octopus-3 (함수당 100개 + full) 정확도 **98.095%**, 지연 **0.38초** (A100, 양자화 없음)

## 준비 (그냥 실행)

In [ ]:
# 랩탑에서 확인한 버전으로 고정 (최신 5.x 는 API 가 바뀌어 에러가 난다)
# 처음 설치한 뒤에는 '런타임 > 세션 다시 시작' 후 이 칸부터 다시 실행
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"   # 4단계에서 학습한 모델
EVAL = "/content/drive/MyDrive/octopus_v2/runs/eval"

---
## 5-1. 우리 모델로 test 800개 생성
**예측**: 전체 정확도는 논문 98.095% 보다 높을까 낮을까? 틀린다면 어느 함수에서?
> (여기에 예측)

처음 3개와 100개마다 생성 결과가 찍힌다. `<nexa_end>` 에서 멈추는지 본다.

In [ ]:
!python -m src.evaluate generate --model {RUN_DIR} --out {EVAL}/ours/preds.jsonl

In [ ]:
!python -m src.evaluate score --preds {EVAL}/ours/preds.jsonl

---
## 5-2. (논문에 없는 추가) 공개 모델 `NexaAIDev/Octopus-v2` 로 같은 test 생성
기준점. 우리 모델이 틀린 문제를 공개 모델도 틀리면 데이터가 어려운 것, 공개 모델만 맞히면 학습 쪽 문제.
공개 모델이 어떤 데이터로 학습했는지는 모르므로 참고용.

In [ ]:
!python -m src.evaluate generate --model NexaAIDev/Octopus-v2 --out {EVAL}/public/preds.jsonl

In [ ]:
!python -m src.evaluate score --preds {EVAL}/public/preds.jsonl

---
## 5-3. 나란히 보기
오답 목록은 `{EVAL}/ours/errors.csv` 를 랩탑에서 열어 `review` 칸에 메모한다.

In [ ]:
import json
from src.evaluate import METRIC_NAMES

rows = {name: json.load(open(f"{EVAL}/{name}/summary.json")) for name in ("ours", "public")}
print(f"{'':<36}{'ours':>12}{'public':>12}")
for sub in ("전체", "seen_in_train=false"):
    print(f"[{sub}]")
    for key, label in METRIC_NAMES.items():
        cells = [rows[n][sub][key] for n in rows]
        print(f"  {label:<34}" + "".join(f"{c:>12}" if key == "n" else f"{c:>12.2%}" for c in cells))
print("[지연 시간, 초]")
for k in ("median", "p90"):
    print(f"  {k:<34}" + "".join(f"{rows[n]['latency_s'][k]:>12.3f}" for n in rows))